# sweet.tv — TV programme recommender (MAP@5)

End-to-end reproduction of the project with the functions in `src/`. The notebook starts from the raw Kaggle CSV
files, rebuilds every intermediate table, re-evaluates every method on the validation protocol and rebuilds the final
submission (v7).

**It writes only to `outputs/nb_check/`** (its own intermediate tables, logs, the rebuilt submission and `checks.json`)
and, at each step, compares its numbers with the stored results in `outputs/*.json`. The last cell fails if any check
does not match, including the md5 of `submissions/submission_v7.csv`.

In [1]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")   # before numpy: ALS needs single-threaded BLAS
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")  # numpy on this Mac is built on Apple Accelerate

import hashlib, json, platform, re, sys, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config

NB = config.OUTPUTS_DIR / "nb_check"
(NB / "interim").mkdir(parents=True, exist_ok=True)
config.INTERIM_DIR = NB / "interim"   # every src function now reads/writes intermediate tables here
config.LOGS_DIR = NB / "logs"

import numpy as np
import pandas as pd
pd.set_option("display.width", 160)

STORED = {n: json.loads((config.OUTPUTS_DIR / f"{n}.json").read_text(encoding=config.ENCODING))
          for n in ["data_audit", "join_stats", "results_validation", "results_final"]}
R = STORED["results_validation"]
CHECKS, TIMES = [], {}

def check(name, got, stored):
    ok = bool(got == stored)
    CHECKS.append({"check": name, "notebook": str(got)[:80], "stored": str(stored)[:80], "ok": ok})
    return ok

def summary(section):
    df = pd.DataFrame([c for c in CHECKS if c["check"].startswith(section)])
    print(f"{section}: {int(df['ok'].sum())} / {len(df)} checks match")
    return df[["check", "notebook", "stored", "ok"]]

print("project root:", ROOT)
print("data dir:", config.DATA_DIR, "exists:", config.DATA_DIR.is_dir())

project root: /Users/kamila/Desktop/MidTerm_Q2/sweettv-recsys
data dir: /Users/kamila/Desktop/MidTerm_Q2/sweettv-tv-program-recommender exists: True


## 1. Data and traps

* `user_id` has up to 20 digits (above int64) → every id is read as a string.
* Schedule dates are `DD.MM.YYYY`, the log is ISO → always parse with an explicit `format=`.
* `tv_show_id = 0` is unlabelled airtime (≈ 11 % of schedule rows carry an id), not an item.
* `duraton` is the session length, not the programme length; 117 rows differ from `stop − start` by exactly 3 600 s —
  sessions across the summer-time switch of 29.03.2020, so the log is in local time.
* Time zone: shifts −3…+3 h and two summer-time variants were tested; no shift is clearly better → `TIME_SHIFT_HOURS = 0`.

In [2]:
t0 = time.time()
from src import audit

views = audit.read_log()
schedules = {p: audit.read_schedule(path) for p, path in config.SCHEDULE_FILES.items()}
a_log = audit.audit_log(views)
a_sched = {p: audit.audit_schedule(df, p) for p, df in schedules.items()}
DA = STORED["data_audit"]
check("1 data: log rows", a_log["rows"], DA["log"]["rows"])
check("1 data: duraton != stop-start rows", a_log["duraton_ne_stop_minus_start_rows"], DA["log"]["duraton_ne_stop_minus_start_rows"])
for p in schedules:
    check(f"1 data: schedule {p} rows", a_sched[p]["rows"], DA["schedule"][p]["rows"])
    check(f"1 data: schedule {p} labelled share", a_sched[p]["labelled_share"], DA["schedule"][p]["labelled_share"])

raw_dates = pd.read_csv(config.SCHEDULE_FILES["11-20"], encoding=config.ENCODING, usecols=["start_time"], dtype=str)["start_time"]
trap = audit.audit_date_trap(raw_dates)
check("1 data: date trap mismatches without format", trap["elementwise_no_format"]["mismatches"],
      DA["date_parsing_trap"]["elementwise_no_format"]["mismatches"])

tz = audit.audit_timezone(views, pd.concat([schedules["11-20"], schedules["21-30"]], ignore_index=True))
check("1 data: time-zone table", tz["table"], DA["timezone"]["table"])
check("1 data: time-zone decision", tz["decision"], DA["timezone"]["decision"])

audit.save_cache(views, schedules)  # -> outputs/nb_check/interim
del views, schedules
TIMES["1 data"] = round(time.time() - t0, 1)
display(pd.DataFrame(tz["table"]).T)
summary("1 data")

2026-10-23 19:46:19,796 | 01_audit | INFO | tz -3h          starts  6.99%  ends  6.99%


2026-10-23 19:46:19,827 | 01_audit | INFO | tz -2h          starts  6.97%  ends  6.87%


2026-10-23 19:46:19,857 | 01_audit | INFO | tz -1h          starts  6.96%  ends  6.76%


2026-10-23 19:46:19,885 | 01_audit | INFO | tz +0h          starts  7.23%  ends  9.01%


2026-10-23 19:46:19,913 | 01_audit | INFO | tz +1h          starts  6.65%  ends  6.48%


...

2026-10-23 19:46:21,219 | 01_audit | INFO | cached schedule_21-30.parquet: 1074394 rows, 13.4 MB


2026-10-23 19:46:21,489 | 01_audit | INFO | cached schedule_31-42.parquet: 1295378 rows, 15.8 MB


,starts_within_120s_pct,ends_within_120s_pct
-3h,6.99,6.99
-2h,6.97,6.87
-1h,6.96,6.76
+0h,7.23,9.01
+1h,6.65,6.48
+2h,6.59,6.63
+3h,6.46,6.43
dst_-2h/-3h,7.02,6.96
dst_+2h/+3h,6.52,6.47


1 data: 11 / 11 checks match


,check,notebook,stored,ok
0,1 data: log rows,5821987,5821987,True
1,1 data: duraton != stop-start rows,117,117,True
2,1 data: schedule 11-20 rows,1074235,1074235,True
3,1 data: schedule 11-20 labelled share,0.1076,0.1076,True
4,1 data: schedule 21-30 rows,1074394,1074394,True
5,1 data: schedule 21-30 labelled share,0.1131,0.1131,True
6,1 data: schedule 31-42 rows,1295378,1295378,True
7,1 data: schedule 31-42 labelled share,0.1118,0.1118,True
8,1 data: date trap mismatches without format,3729,3729,True
9,1 data: time-zone table,{'-3h': {'starts_within_120s_pct': np.float64(...,"{'-3h': {'starts_within_120s_pct': 6.99, 'ends...",True


## 2. Join: sessions → (user, airing) interactions

An airing is a labelled schedule row (`tv_show_id != 0`, end = start + duration). Overlapping sessions of one user on
one channel are merged first; the overlap with each airing on the same channel is summed per (user, airing), clipped
to the airing's duration; `screen_time = overlap / duration`, `watched = screen_time ≥ 0.8` (as in the Kaggle answer).
The search is vectorised with `searchsorted` on sorted arrays.

In [3]:
t0 = time.time()
from src import join

I = config.INTERIM_DIR
log_df = pd.read_parquet(I / "log.parquet")
sch = {p: join.canonical_ids(pd.read_parquet(I / f"schedule_{p}.parquet")) for p in config.SCHEDULE_FILES}
train_sched, _ = join.dedup_schedule([sch["11-20"], sch["21-30"]])
sessions = join.merge_sessions(log_df, shift_s=config.TIME_SHIFT_HOURS * 3600)
av_full = join.airing_views(sessions, join.build_airings(train_sched))
items_full, _ = join.build_items(join.dedup_schedule(list(sch.values()))[0])
av_full.to_parquet(I / "airing_views.parquet", index=False)
items_full.to_parquet(I / "items.parquet", index=False)

JS = STORED["join_stats"]
check("2 join: sessions after merge", len(sessions), JS["sessions"]["after_merge"])
check("2 join: (user, airing) pairs", len(av_full), JS["pairs"]["total"])
check("2 join: watched pairs", int(av_full["watched"].sum()), JS["pairs"]["watched"])
for t, n in JS["watched_pairs_by_threshold"].items():
    check(f"2 join: pairs with screen_time >= {t}", int((av_full["screen_time"] >= float(t)).sum()), n)
check("2 join: items", len(items_full), JS["items"]["rows"])
del log_df, sch, train_sched, sessions, av_full
TIMES["2 join"] = round(time.time() - t0, 1)
summary("2 join")

2 join: 7 / 7 checks match


,check,notebook,stored,ok
0,2 join: sessions after merge,5725415,5725415,True
1,"2 join: (user, airing) pairs",3014129,3014129,True
2,2 join: watched pairs,611760,611760,True
3,2 join: pairs with screen_time >= 0.5,938140,938140,True
4,2 join: pairs with screen_time >= 0.8,611760,611760,True
5,2 join: pairs with screen_time >= 0.9,521370,521370,True
6,2 join: items,9671,9671,True


## 3. Protocol and MAP@5

* Validation: train on ISO weeks 11–20, ground truth = a user's top-5 programmes by watched airings in weeks 21–30.
  Final: train on 11–30, predict 31–42.
* Cohort A = submission users with ≥ 5 truth programmes; cohort C = all log users with ≥ 5.
* Pool = labelled programmes airing in the target window; repeats allowed; cold start and short lists are topped up
  from popularity → exactly 5 distinct ids.
* MAP@5 as in `ml_metrics`: AP is divided by `min(len(actual), 5)`.

In [4]:
t0 = time.time()
from src import protocol as P
from src.metrics import apk

av, items = P.load_airing_views(), P.load_items()
acc = P.data_accounting(av, items)
check("3 protocol: data accounting", acc, R["protocol"]["data_accounting"])
truth = P.build_truth(av, "val")
cohorts = P.build_cohorts(truth, P.submission_users())
users = sorted(set(cohorts["A"]) | set(cohorts["C"]))
train, pool = P.train_views(av, "val"), P.candidate_pool(items, "val")
print("cohort A:", len(cohorts["A"]), "| cohort C:", len(cohorts["C"]), "| pool:", len(pool))
print("AP@5 example, actual [1, 2], predicted [2, 6, 7, 8, 1]:", apk(["1", "2"], ["2", "6", "7", "8", "1"]))
TIMES["3 protocol"] = round(time.time() - t0, 1)
summary("3 protocol")

cohort A: 1150 | cohort C: 4030 | pool: 6376
AP@5 example, actual [1, 2], predicted [2, 6, 7, 8, 1]: 0.7
3 protocol: 1 / 1 checks match


,check,notebook,stored,ok
0,3 protocol: data accounting,"{'cohort_sizes': {'A': 1150, 'C': 4030, 'C_sam...","{'cohort_sizes': {'A': 1150, 'C': 4030, 'C_sam...",True


## 4. Popularity baseline (and personal history for reference)

In [5]:
t0 = time.time()
from src import baselines

PREDS, NB_MAP = {}, {}
for name, fn in baselines.METHODS.items():
    PREDS[name], _ = fn(train, pool, users)
    NB_MAP[name] = P.evaluate(PREDS[name], truth, cohorts)
    check(f"4 baselines: {name} MAP@5 A", NB_MAP[name]["map@5_A"], R["methods"][name]["map@5_A"])
TIMES["4 baselines"] = round(time.time() - t0, 1)
summary("4 baselines")

4 baselines: 2 / 2 checks match


,check,notebook,stored,ok
0,4 baselines: popularity MAP@5 A,0.235699,0.235699,True
1,4 baselines: personal_history MAP@5 A,0.343832,0.343832,True


## 5. Personalized PageRank on the co-view graph

`W = XᵀX` over watched interactions (diagonal 0, top-50 neighbours per row, row-normalised); restart = the user's
normalised history; `p = α(Pᵀp + d·v) + (1 − α)v` with dangling mass `d`, tol 1e-6.

In [6]:
t0 = time.time()
from src import ppr

graph = ppr.fit_graph(train)
popular = P.popularity_ranking(train, pool)
for a in ppr.ALPHAS:
    preds, _, conv = ppr.recommend(graph, pool, users, a, popular)
    check(f"5 ppr: alpha {a} MAP@5 A", P.evaluate(preds, truth, cohorts)["map@5_A"], R["methods"]["ppr"]["grid"][str(a)]["map@5_A"])
    if a == R["methods"]["ppr"]["best_alpha"]:
        PREDS["ppr"], NB_MAP["ppr"] = preds, P.evaluate(preds, truth, cohorts)
print("graph:", graph["stats"])
TIMES["5 ppr"] = round(time.time() - t0, 1)
summary("5 ppr")

graph: {'items': 2499, 'edges_before_top_k': 1217866, 'edges_after_top_k': 120654, 'dangling_items': 0, 'max_abs_row_sum_minus_1': 3.3306690738754696e-16, 'dangling_row_sum_max': 0.0}
5 ppr: 6 / 6 checks match


,check,notebook,stored,ok
0,5 ppr: alpha 0.1 MAP@5 A,0.353838,0.353838,True
1,5 ppr: alpha 0.2 MAP@5 A,0.354325,0.354325,True
2,5 ppr: alpha 0.3 MAP@5 A,0.354913,0.354913,True
3,5 ppr: alpha 0.5 MAP@5 A,0.354304,0.354304,True
4,5 ppr: alpha 0.7 MAP@5 A,0.338701,0.338701,True
5,5 ppr: alpha 0.85 MAP@5 A,0.305641,0.305641,True


## 6. Matrix factorization (implicit ALS)

User × item matrix of watched-airing counts; scores computed as `user_factors @ item_factors.T` over the pool
(already-watched programmes are kept). Grid: factors 64 × regularization {0.01, 0.1} × alpha {10, 40}, 15 iterations,
CPU, one thread, `random_state=42`. ALS stays below popularity on this data.

In [7]:
t0 = time.time()
import itertools
from src import mf

M, users_index, mf_items = mf.user_item_matrix(train)
for factors, reg, alpha in itertools.product(*mf.GRID.values()):
    key = f"f{factors}_reg{reg}_a{alpha:g}"
    model = mf.fit_als(M, factors, reg, alpha)
    preds, _ = mf.recommend(model.user_factors, model.item_factors, users_index, mf_items, pool, users, popular)
    check(f"6 mf: {key} MAP@5 A", P.evaluate(preds, truth, cohorts)["map@5_A"], R["methods"]["mf"]["grid"][key]["map@5_A"])
    if key == R["methods"]["mf"]["best"]:
        PREDS["mf"], NB_MAP["mf"], best_model = preds, P.evaluate(preds, truth, cohorts), model
again = mf.fit_als(M, **{k: R["methods"]["mf"]["grid"][R["methods"]["mf"]["best"]][k] for k in ("factors", "regularization", "alpha")})
check("6 mf: best config refit is bit-identical", bool(np.array_equal(best_model.item_factors, again.item_factors)), True)
TIMES["6 mf"] = round(time.time() - t0, 1)
summary("6 mf")

6 mf: 5 / 5 checks match


,check,notebook,stored,ok
0,6 mf: f64_reg0.01_a10 MAP@5 A,0.167461,0.167461,True
1,6 mf: f64_reg0.01_a40 MAP@5 A,0.101223,0.101223,True
2,6 mf: f64_reg0.1_a10 MAP@5 A,0.166736,0.166736,True
3,6 mf: f64_reg0.1_a40 MAP@5 A,0.106058,0.106058,True
4,6 mf: best config refit is bit-identical,True,True,True


## 7. Content-based filtering

Item vector = one-hot category, multi-hot genres 1–3 and TF-IDF over directors and actors (one person = one token);
each block L2-normalised. Profile = recency-weighted mean of the user's watched programmes; score = cosine.

In [8]:
t0 = time.time()
from src import cbf

V, item_ids, vec_stats = cbf.item_matrix(items)
H_V2 = float(R["improve"]["ppr"]["best"]["half_life"])
ranked, _ = cbf.rank_users(V, item_ids, train, H_V2, P.SPLITS["val"]["train_weeks"][1], pool, users)
PREDS["cbf"], _ = P.fill_to_k(ranked, users, popular, pool)
NB_MAP["cbf"] = P.evaluate(PREDS["cbf"], truth, cohorts)
check("7 cbf: MAP@5 A", NB_MAP["cbf"]["map@5_A"], R["methods"]["cbf"]["map@5_A"])
print("vector blocks:", vec_stats)
TIMES["7 cbf"] = round(time.time() - t0, 1)
summary("7 cbf")

vector blocks: {'category': {'features': 8, 'programmes_with_any': 9671}, 'genres': {'features': 65, 'programmes_with_any': 9440}, 'directors': {'features': 4792, 'programmes_with_any': 9488}, 'actors': {'features': 12338, 'programmes_with_any': 9524}}
7 cbf: 1 / 1 checks match


,check,notebook,stored,ok
0,7 cbf: MAP@5 A,0.207052,0.207052,True


## 8. Hybrids and improvements of PPR

* **PPR + MF by reciprocal rank fusion** — any MF weight lowers MAP; the best weight is pure PPR.
* **v2** = PPR × (airings of the programme in the target window)^0.25 + recency weight 0.5^(Δweeks / 4).
* **v5** = v2 trained on soft weights (screen_time of every pair with screen_time ≥ 0.1) instead of the 0.8 threshold.
* **v6** = v5 + channel × day × hour habits: score = slot_ui / max + 2 · v5 / max.

Expected-frequency re-ranking, CBF hybrids, binary labels and ensembles did not help (see `results_validation.json`
and `outputs/design_choices.md`).

In [9]:
t0 = time.time()
from src import hybrid, improve, signal_exp, slots

# PPR + MF (RRF)
alpha = R["methods"]["ppr"]["best_alpha"]
ranked_ppr, _ = ppr.rank_users(graph, pool, users, alpha, depth=hybrid.DEPTH)
ranked_mf = mf.rank_users(best_model.user_factors, best_model.item_factors, users_index, mf_items, pool, users, depth=hybrid.DEPTH)
for w in hybrid.WEIGHTS:
    preds, _ = P.fill_to_k(hybrid.rrf(ranked_ppr, ranked_mf, users, w), users, popular, pool)
    check(f"8 hybrids: RRF w={w:g} MAP@5 A", P.evaluate(preds, truth, cohorts)["map@5_A"], R["methods"]["hybrid_rrf"]["grid"][str(w)]["map@5_A"])

# v2, v5, v6
v2 = R["improve"]["ppr"]["best"]
beta, half_life = v2["beta"], float(v2["half_life"])
PREDS["v2"], _, _ = improve.Runner(av, items, "val", users, alpha).predict(beta, half_life)
check("8 hybrids: v2 MAP@5 A", P.evaluate(PREDS["v2"], truth, cohorts)["map@5_A"], v2["map@5_A"])

av5 = slots.load_views()  # airing_views with screen_time, channel and airing start
v5cfg = {k: R["improve_signal"]["v2_signal"]["final_v5"][k] for k in ("beta", "half_life", "mode", "t")}
PREDS["v5"], _, _ = signal_exp.make_runner(av5, items, "val", users, alpha, v5cfg["mode"], v5cfg["t"]).predict(beta, half_life)
check("8 hybrids: v5 MAP@5 A", P.evaluate(PREDS["v5"], truth, cohorts)["map@5_A"],
      R["improve_signal"]["v2_signal"]["table"]["soft_0.1"]["map@5_A"])

s_res = R["improve_slots"]["v5_plus_slots"]
lam = float(s_res["table"][s_res["best"]]["lambda"])
run6, _, v5_r, v5_s, s_r, s_s, _, _ = slots.build(av5, items, "val", users, alpha, v5cfg)
PREDS["v6"], _ = P.fill_to_k(slots.fuse(v5_r, v5_s, s_r, s_s, lam), users, run6.popular, run6.pool)
NB_MAP["v6"] = P.evaluate(PREDS["v6"], truth, cohorts)
check("8 hybrids: v6 MAP@5 A", NB_MAP["v6"]["map@5_A"], s_res["table"][s_res["best"]]["map@5_A"])
TIMES["8 hybrids"] = round(time.time() - t0, 1)
summary("8 hybrids")

8 hybrids: 8 / 8 checks match


,check,notebook,stored,ok
0,8 hybrids: RRF w=0 MAP@5 A,0.167461,0.167461,True
1,8 hybrids: RRF w=0.25 MAP@5 A,0.232206,0.232206,True
2,8 hybrids: RRF w=0.5 MAP@5 A,0.308638,0.308638,True
3,8 hybrids: RRF w=0.75 MAP@5 A,0.346348,0.346348,True
4,8 hybrids: RRF w=1 MAP@5 A,0.354913,0.354913,True
5,8 hybrids: v2 MAP@5 A,0.366742,0.366742,True
6,8 hybrids: v5 MAP@5 A,0.377609,0.377609,True
7,8 hybrids: v6 MAP@5 A,0.383484,0.383484,True


## 9. Learned re-ranking (v7)

Candidates per user: v6 top-50 ∪ slot top-50 ∪ the user's pool history (≤ 100). 16 features, all window-length
dependent ones per week. Label 5…1 for truth places 1–5, else 0. `LGBMRanker` (lambdarank, 300 trees, 31 leaves,
learning rate 0.05, seed 42, deterministic, one thread). Evaluation out-of-fold over two folds of cohort-C users.

In [10]:
t0 = time.time()
from src import rerank

C = cohorts["C"]
win = rerank.build_window(av5, items, "val", C, alpha, v5cfg, lam, V, item_ids)
df = rerank.add_labels(win["df"], truth)
perm = np.random.default_rng(config.SEED).permutation(np.array(sorted(C)))
folds = [set(perm[: len(perm) // 2]), set(perm[len(perm) // 2:])]
oof, models = {}, []
for k in range(2):
    model = rerank.fit(df[df["user_id"].isin(folds[k])])
    models.append(model)
    test_df = df[df["user_id"].isin(folds[1 - k])]
    oof.update(rerank.rank_by(test_df, model.predict(test_df[rerank.FEATURES]), sorted(folds[1 - k]), win["run"]))
PREDS["v7"], NB_MAP["v7"] = oof, P.evaluate(oof, truth, cohorts)
check("9 rerank: v7 OOF MAP@5 A", NB_MAP["v7"]["map@5_A"], R["rerank"]["lgbm_v6"]["oof"]["map@5_A"])
check("9 rerank: v7 OOF MAP@5 C", NB_MAP["v7"]["map@5_C"], R["rerank"]["lgbm_v6"]["oof"]["map@5_C"])
imp = rerank.importance(models)["gain"]
print("candidates:", win["stats"]["candidates"], "| top features by gain:", list(imp)[:5])
TIMES["9 rerank"] = round(time.time() - t0, 1)
summary("9 rerank")

candidates: 383049 | top features by gain: ['v6_rank', 'v6_score', 'v5_ppr', 'slot_norm', 'n_airings_target_pw']
9 rerank: 2 / 2 checks match


,check,notebook,stored,ok
0,9 rerank: v7 OOF MAP@5 A,0.393562,0.393562,True
1,9 rerank: v7 OOF MAP@5 C,0.354267,0.354267,True


## 10. Comparison (validation, MAP@5)

In [11]:
rows = [("popularity", NB_MAP["popularity"]), ("personal_history (reference)", NB_MAP["personal_history"]),
        ("PPR v1 (alpha 0.3)", NB_MAP["ppr"]), ("MF (ALS)", NB_MAP["mf"]), ("CBF", NB_MAP["cbf"]),
        ("v2", P.evaluate(PREDS["v2"], truth, cohorts)), ("v5", P.evaluate(PREDS["v5"], truth, cohorts)),
        ("v6", NB_MAP["v6"]), ("v7 (OOF)", NB_MAP["v7"])]
comparison = pd.DataFrame([{"method": n, "MAP@5 A": m["map@5_A"], "MAP@5 C": m["map@5_C"]} for n, m in rows])
comparison

,method,MAP@5 A,MAP@5 C
0,popularity,0.235699,0.213795
1,personal_history (reference),0.343832,0.304210
2,PPR v1 (alpha 0.3),0.354913,0.317993
3,MF (ALS),0.167461,0.161546
4,CBF,0.207052,0.182445
5,v2,0.366742,0.329417
6,v5,0.377609,0.341646
7,v6,0.383484,0.346600
8,v7 (OOF),0.393562,0.354267


## 11. Final submission (v7)

The re-ranker is trained on all cohort-C users (features weeks 11–20, labels 21–30) and applied to features of weeks
11–30 with the pool of weeks 31–42. The rebuilt file must be byte-identical to `submissions/submission_v7.csv`.
**Grading uses the best Kaggle private score; public is reference only.**

In [12]:
t0 = time.time()
from src.submission import validate_submission, write_submission

final_model = rerank.fit(df)
sub_users = P.submission_users()
test_win = rerank.build_window(av5, items, "test", sub_users, alpha, v5cfg, lam, V, item_ids)
preds_final = rerank.rank_by(test_win["df"], final_model.predict(test_win["df"][rerank.FEATURES]), sub_users, test_win["run"])
nb_sub = NB / "submission_v7.csv"
write_submission(preds_final, nb_sub)
errors = validate_submission(nb_sub, test_win["run"].pool)
md5 = lambda p: hashlib.md5(Path(p).read_bytes()).hexdigest()
check("11 final: rebuilt submission passes validation", errors, [])
check("11 final: md5 = submissions/submission_v7.csv", md5(nb_sub), md5(config.SUBMISSIONS_DIR / "submission_v7.csv"))
check("11 final: md5 = results_final.json", md5(nb_sub), STORED["results_final"]["md5"])
TIMES["11 final"] = round(time.time() - t0, 1)
kaggle = pd.DataFrame(STORED["results_final"]["kaggle_submissions"]["submissions"])[["name", "kaggle_private", "validation_map@5_A", "kaggle_public"]]
print("Kaggle scores as reported by the author (private = graded, public = reference):")
display(kaggle)
summary("11 final")

Kaggle scores as reported by the author (private = graded, public = reference):


,name,kaggle_private,validation_map@5_A,kaggle_public
0,final (v1),0.39806,0.354913,0.40746
1,v2,0.39944,0.366742,0.40299
2,v5,0.41627,0.377609,0.42955
3,v6,0.42440,0.383484,0.43613
4,v7,0.43671,0.393562,0.43802


11 final: 3 / 3 checks match


,check,notebook,stored,ok
0,11 final: rebuilt submission passes validation,[],[],True
1,11 final: md5 = submissions/submission_v7.csv,d0e51925c3911a9174817c3de129e94c,d0e51925c3911a9174817c3de129e94c,True
2,11 final: md5 = results_final.json,d0e51925c3911a9174817c3de129e94c,d0e51925c3911a9174817c3de129e94c,True


## 12. Two example users

The most and the least active cohort-A users of weeks 11–20 (`outputs/two_users.md`); their AP@5 per method is
recomputed from this notebook's predictions.

In [13]:
text = (config.OUTPUTS_DIR / "two_users.md").read_text(encoding=config.ENCODING)
example_users = re.findall(r"^## [^:]+: `(\d+)`", text, flags=re.M)
summary_rows = [l for l in text.splitlines() if l.startswith("| Самый")]
method_keys = ["popularity", "ppr", "mf", "cbf", "v6", "v7"]
for u, line in zip(example_users, summary_rows):
    stored_ap = [float(x) for x in line.strip("|").split("|")[1:]]
    nb_ap = [round(apk(truth[u], PREDS[m][u]), 4) for m in method_keys]
    check(f"12 two users: AP@5 of {u}", nb_ap, stored_ap)
print(pd.DataFrame({u: [round(apk(truth[u], PREDS[m][u]), 4) for m in method_keys] for u in example_users}, index=method_keys))
summary("12 two users")

            10500740270951698446  10120801182216576239
popularity                  0.20                   0.2
ppr                         0.76                   0.2
mf                          0.00                   0.2
cbf                         0.20                   0.2
v6                          0.60                   0.2
v7                          0.60                   0.2
12 two users: 2 / 2 checks match


,check,notebook,stored,ok
0,12 two users: AP@5 of 10500740270951698446,"[0.2, 0.76, 0.0, 0.2, 0.6, 0.6]","[0.2, 0.76, 0.0, 0.2, 0.6, 0.6]",True
1,12 two users: AP@5 of 10120801182216576239,"[0.2, 0.2, 0.2, 0.2, 0.2, 0.2]","[0.2, 0.2, 0.2, 0.2, 0.2, 0.2]",True


## 13. Reproducibility

In [14]:
import importlib.metadata as md_
versions = {p: md_.version(p) for p in ["pandas", "numpy", "scipy", "pyarrow", "implicit", "lightgbm", "scikit-learn", "psutil"]}
env = {"python": platform.python_version(), "executable": sys.executable, "platform": platform.platform(),
       "seed": config.SEED, "versions": versions, "section_runtime_s": TIMES}
checks = pd.DataFrame(CHECKS)
(NB / "checks.json").write_text(json.dumps({"environment": env, "checks": CHECKS}, ensure_ascii=False, indent=2), encoding=config.ENCODING)
print(json.dumps(env, indent=1))
print(f"checks: {int(checks['ok'].sum())} / {len(checks)} match")
assert checks["ok"].all(), checks[~checks["ok"]]

{
 "python": "3.12.7",
 "executable": "/Users/kamila/Desktop/MidTerm_Q2/sweettv-recsys/.venv/bin/python",
 "platform": "macOS-15.7.4-arm64-arm-64bit",
 "seed": 42,
 "versions": {
  "pandas": "3.0.6",
  "numpy": "2.5.4",
  "scipy": "1.18.1",
  "pyarrow": "26.0.0",
  "implicit": "0.7.3",
  "lightgbm": "4.7.0",
  "scikit-learn": "1.9.1",
  "psutil": "7.2.2"
 },
 "section_runtime_s": {
  "1 data": 17.5,
  "2 join": 5.4,
  "3 protocol": 1.4,
  "4 baselines": 0.2,
  "5 ppr": 7.6,
  "6 mf": 5.4,
  "7 cbf": 1.5,
  "8 hybrids": 11.2,
  "9 rerank": 20.5,
  "11 final": 15.6
 }
}
checks: 48 / 48 match
